In [1]:
import pandas as pd
from snownlp import SnowNLP
import json

In [2]:
# 使用大模型前先过滤敏感词
model_name = 'qwen-plus-latest'
sen_names = pd.read_csv('llm_sensitive_name.csv')['name'].tolist()

## Agent 1: Text summarization task

该任务既可以使用LLM直接输出结果，也可以通过batch方式；batch方式可以节约token成本；SUMM-Agent采用Batch推理。

In [3]:
def LLM_builder_gen_title(content):
    for i in sen_names:
        content = content.replace(i, '某领导人')
        
    role = '你是文本摘要助手，擅长提炼新闻文章中和澳门相关的信息，并根据该内容写一段不超过500字的总结。'    
    prompt = f'''
    请根据新闻文章中澳门相关的内容，写一段不超过500字的总结。
    **任务要求**：
    1. 不要深度揣测，内容只需要符合客观实际即可。
    2. 只能输出结果，不允许提供任何解释和推理过程的回答。
    **新闻文章内容如下：**：
    {content}

    请根据以上内容，完成任务。
    '''
    messages = [
        {"role": "system", "content": role},
        {"role": "user", "content": prompt.strip()}
    ]
    
    return messages

In [4]:
def builder_summ(df, filename):
    with open('./data/llm-gen-summ/' + filename + model_name + '.jsonl', 'w', encoding='utf-8') as fout:
        for i, row in df.iterrows():
            text = row['text']
            for name in sen_names:
                text = text.replace(name, '  ')
            llm_bu = LLM_builder_gen_title(text)
            body = {'model': model_name, 'messages': llm_bu, 'temperature': 0.1}
            request = {'custom_id': row['idx'], 'method': 'POST', 'url': '/v1/chat/completions', 'body': body}
            fout.write(json.dumps(request, separators=(',', ':'), ensure_ascii=False) + '\n')

In [5]:
# local_media = pd.read_csv('./data/woa_macau.csv')
# local_media['idx'] = range(len(local_media))
# builder_summ(local_media, 'local_media_')

In [6]:
other_media = pd.read_csv('./data/woa_other_cities.csv')
other_media['idx'] = range(len(other_media))
builder_summ(other_media, 'other_media_')